# Invoke Bedrock model for code generation

> *This notebook should work well with the **`Python 3`** kernel in SageMaker Studio*

> **Updated November 2026.** Amazon Titan Text, Anthropic Claude 3.x and Stable Diffusion XL reached end of life on Amazon Bedrock (see [Model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). This notebook now uses [Amazon Nova 2 Lite](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html) through the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) and [Claude Haiku 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html) / [Claude Sonnet 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-5-5.html) through cross-Region inference profiles. Claude 4.7+ no longer accepts `temperature`, `top_p` or `top_k` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), so those parameters were removed from the Claude requests.

## Introduction

In this notebook we show you how to use a LLM to generate code based on the text prompt. We will use Anthropic Claude Sonnet 5.5 using the Boto3 API. 

The prompt used in this example is called a zero-shot prompt because we are not providing any examples of text other than the prompt.

**Note:** *This notebook can be run within or outside of AWS environment.*

#### Context
To demonstrate the code generation capability of Amazon Bedrock, we will explore the use of Boto3 client to communicate with Amazon Bedrock API. We will demonstrate different configurations available as well as how simple input can lead to desired outputs. We will explore code generation for two use cases:
1. Python code generation for analytical QnA
2. SQL query generation

#### Pattern
In both use cases, we will simply provide the Amazon Bedrock API with an input consisting of a task, an instruction and an input for the model under the hood to generate an output without providing any additional example. The purpose here is to demonstrate how the powerful LLMs easily understand the task at hand and generate compelling outputs.

![](../imgs/bedrock-code-gen.png)

## Use case 1 - Python code generation for Analytical QnA
To demonstrate the generation capability of models in Amazon Bedrock, let's take the use case of code generation with Python to do some basic analytical QnA.

#### Persona

You are Moe, a Data Analyst, at AnyCompany. The company wants to understand its sales performance for different products for different products over the past year. You have been provided a dataset named sales.csv. The dataset contains the following columns:

- Date (YYYY-MM-DD) format
- Product_ID (unique identifer for each product)
- Price (price at which each product was sold)

#### Implementation
To fulfill this use case, in this notebook we will show how to generate code for a given prompt. We will use the Anthropic Claude v3 using the Amazon Bedrock API with Boto3 client. 

## Setup


In [1]:
import json
import os
import sys

import boto3
import botocore

boto3_bedrock = boto3.client('bedrock-runtime')

## Code Generation

Following on the use case explained above, let's prepare an input for  the Amazon Bedrock service to generate python program for our use-case.

#### Lab setup - create sample sales.csv data for this lab.



In [2]:
# create sales.csv file
import csv

data = [
    ["date", "product_id", "price", "units_sold"],
    ["2023-01-01", "P001", 50, 20],
    ["2023-01-02", "P002", 60, 15],
    ["2023-01-03", "P001", 50, 18],
    ["2023-01-04", "P003", 70, 30],
    ["2023-01-05", "P001", 50, 25],
    ["2023-01-06", "P002", 60, 22],
    ["2023-01-07", "P003", 70, 24],
    ["2023-01-08", "P001", 50, 28],
    ["2023-01-09", "P002", 60, 17],
    ["2023-01-10", "P003", 70, 29],
    ["2023-02-11", "P001", 50, 23],
    ["2023-02-12", "P002", 60, 19],
    ["2023-02-13", "P001", 50, 21],
    ["2023-02-14", "P003", 70, 31],
    ["2023-03-15", "P001", 50, 26],
    ["2023-03-16", "P002", 60, 20],
    ["2023-03-17", "P003", 70, 33],
    ["2023-04-18", "P001", 50, 27],
    ["2023-04-19", "P002", 60, 18],
    ["2023-04-20", "P003", 70, 32],
    ["2023-04-21", "P001", 50, 22],
    ["2023-04-22", "P002", 60, 16],
    ["2023-04-23", "P003", 70, 34],
    ["2023-05-24", "P001", 50, 24],
    ["2023-05-25", "P002", 60, 21]
]

# Write data to sales.csv
with open('sales.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(data)

print("sales.csv has been created!")

sales.csv has been created!


#### Analyzing sales with Amazon Bedrock generated Python program

In [3]:
# Create the prompt
# Analyzing sales

prompt_data = """
You have a CSV, sales.csv, with columns:
- date (YYYY-MM-DD)
- product_id
- price
- units_sold

Create a python program to analyze the sales data from a CSV file. The program should be able to read the data, and determine below:

- Total revenue for the year
- The product with the highest revenue
- The date with the highest revenue
- Visualize monthly sales using a bar chart

Ensure the code is syntactically correct, bug-free, optimized, not span multiple lines unnessarily, and prefer to use standard libraries. Return only python code without any surrounding text, explanation or context.
Do not use pandas library for the solution.
"""

Let's start by using the Anthropic Claude Sonnet 5.5 model ([model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-5-5.html)). Claude 4.7+ does not accept `temperature`, `top_k` or `top_p` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), so the request body only sets `max_tokens`. Claude 5.5 has [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) on by default: the response can start with a `thinking` block (Invoke) or a `reasoningContent` block (Converse) before the `text` block, so the code below picks the text blocks explicitly instead of reading `content[0]`. These demos ask for short answers with a small token budget, so [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) is turned off explicitly with `"thinking": {"type": "disabled"}`; otherwise the model can spend the whole `max_tokens` budget thinking and return no text. Claude Sonnet 5.5 rejects `"disabled"` and uses `"thinking": {"type": "between_tools"}` instead, which without tools returns only text (see the per-model table in the [thinking docs](https://platform.claude.com/docs/en/build-with-claude/thinking)).

In [4]:
body = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 4096,
    "thinking": {"type": "between_tools"},
    "messages": [
        {
            "role": "user",
            "content": [{"type": "text", "text": prompt_data}]
        }
    ],
})

#### Invoke the Anthropic Claude Sonnet 5.5 model

In [5]:
from IPython.display import clear_output, display, display_markdown, Markdown
modelId = "global.anthropic.claude-sonnet-5-5"
accept = 'application/json'
contentType = 'application/json'

response = boto3_bedrock.invoke_model(body=body, modelId=modelId, accept=accept, contentType=contentType)
response_body = json.loads(response.get('body').read())

generated_text = "".join(block["text"] for block in response_body["content"] if block["type"] == "text")
print(generated_text)

```python
import csv
from collections import defaultdict
from datetime import datetime

import matplotlib.pyplot as plt


def load_sales(path):
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try:
                date = datetime.strptime(row["date"].strip(), "%Y-%m-%d").date()
                price = float(row["price"])
                units = float(row["units_sold"])
            except (ValueError, KeyError, AttributeError):
                continue
            yield date, row["product_id"].strip(), price * units


def analyze(path):
    total = 0.0
    by_product = defaultdict(float)
    by_date = defaultdict(float)
    by_month = defaultdict(float)

    for date, product, revenue in load_sales(path):
        total += revenue
        by_product[product] += revenue
        by_date[date] += revenue
        by_month[(date.year, date.month)] += revenue

    if not by_product:
        print("No valid sales data found.")
        ret

#### (Optional) Execute the Bedrock generated code for validation. Go to text editor to copy the generated code as printed output can be trucncated. Replace the code in below cell.

In [6]:
# Sample Generated Python Code ( Generated with Amazon Bedrock in previous step)
%pip install -q matplotlib

import csv
from collections import defaultdict
import matplotlib.pyplot as plt

revenue = 0
monthly_revenue = defaultdict(int)
product_revenue = defaultdict(int)
max_revenue = 0
max_revenue_date = ''
max_revenue_product = ''

with open('sales.csv') as f:
    reader = csv.reader(f)
    next(reader)
    for row in reader:
        date = row[0]
        product = row[1]
        price = float(row[2])
        units = int(row[3])

        revenue += price * units
        product_revenue[product] += price * units
        monthly_revenue[date[:7]] += price * units

        if revenue > max_revenue:
            max_revenue = revenue
            max_revenue_date = date
            max_revenue_product = product

months = list(monthly_revenue.keys())
values = list(monthly_revenue.values())

plt.bar(months, values)
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.title('Monthly Revenue')
plt.show()

print('Total Revenue:', revenue)
print('Product with max revenue:', max_revenue_product)
print('Date with max revenue:', max_revenue_date)

___

## Use case 2 - SQL query generation

To demonstrate the generation capability of models in Amazon Bedrock, let's take the use case of code generation with Python to do some basic analytical QnA.

In this section we show you how to use a LLM to generate SQL Query to analyze Sales data.

We will use Amazon Nova 2 Lite through the Converse API. 

The prompt used in this example is called a zero-shot prompt because we are not providing any examples of text other than the prompt.

**Note:** *This notebook can be run within or outside of AWS environment.*

#### Context
To demonstrate the SQL code generation capability of Amazon Bedrock, we will explore the use of Boto3 client to communicate with Amazon Bedrock API. We will demonstrate different configurations available as well as how simple input can lead to desired outputs.

#### Pattern
We will simply provide the Amazon Bedrock API with an input consisting of a task, an instruction and an input for the model under the hood to generate an output without providing any additional example. The purpose here is to demonstrate how the powerful LLMs easily understand the task at hand and generate compelling outputs.

#### Use case
Let's take the use case to generate SQL queries to analyze sales data, focusing on trends, top products and average sales.

#### Persona
Maya is a business analyst, at AnyCompany primarily focusing on sales and inventory data. She is transitioning from Speadsheet analysis to data-driven analysis and want to use SQL to fetch specific data points effectively. She wants to use LLMs to generate SQL queries for her analysis. 

#### Implementation
To fulfill this use case, in this notebook we will show how to generate SQL queries. We will use the Anthropic Claude v3 model using the Amazon Bedrock API with Boto3 client. 

### Generate SQL Query

Following on the use case explained above, let's prepare an input for  the Amazon Bedrock service to generate SQL query.

In [7]:
# create the prompt to generate SQL query
prompt_data = """

Human: AnyCompany has a database with a table named sales_data containing sales records. The table has following columns:
- date (YYYY-MM-DD)
- product_id
- price
- units_sold

Can you generate SQL queries for the below: 
- Identify the top 5 best selling products by total sales for the year 2023
- Calculate the monthly average sales for the year 2023

Assistant:
"""


Let's use the Amazon Nova 2 Lite model ([model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html)) through the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html):

In [8]:
# Converse API: messages + inference parameters

messages = [{"role": "user", "content": [{"text": prompt_data}]}]
inference_config = {
    "maxTokens": 512,
    "stopSequences": [],
    "temperature": 0.5,
    "topP": 0.9,
}

In [9]:
modelId = "us.amazon.nova-2-lite-v1:0"

response = boto3_bedrock.converse(modelId=modelId, messages=messages, inferenceConfig=inference_config)

print(f"Token count: {response['usage']['outputTokens']}")
print(f"Output text: {response['output']['message']['content'][0]['text']}")
print(f"Completion reason: {response['stopReason']}")

Token count: 512
Output text: ### SQL Queries for **AnyCompany** `sales_data` Table

Below are the SQL queries to address your requirements using the `sales_data` table with the columns:  
- `date` (YYYY-MM-DD)  
- `product_id`  
- `price`  
- `units_sold`

---

## ✅ **1. Identify the Top 5 Best-Selling Products by Total Sales for the Year 2023**

### **Explanation:**
We need to calculate **total sales** for each product in 2023.  
Total sales = `price` × `units_sold`

We will:
- Filter records from the year **2023**
- Group by `product_id`
- Calculate total sales per product
- Order by total sales in **descending** order
- Limit to the **top 5**

### **SQL Query:**

```sql
SELECT
    product_id,
    SUM(price * units_sold) AS total_sales
FROM
    sales_data
WHERE
    strftime('%Y', date) = '2023'  -- For SQLite
    -- OR
    -- EXTRACT(YEAR FROM date) = 2023  -- For PostgreSQL
    -- OR
    -- YEAR(date) = 2023  -- For MySQL
GROUP BY
    product_id
ORDER BY
    total_sales DESC
LIMIT 

## Conclusion
You have now experimented with using `boto3` SDK which provides a vanilla exposure to Amazon Bedrock API. Using this API you generate a python program to analyze and visualize given sales data, and generate SQL statements based on an input task and schema.

### Take aways
- Adapt this notebook to experiment with different models available through Amazon Bedrock such as Amazon Nova and AI21 Labs Jurassic models!
